In [3]:
import os
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal

# Define output directory for spectrogram files and create it if not exists
output_dir = 'spectrogram_output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def load_audio_and_annotation(file_base_name):
    # Load the WAV audio file
    audio_path = file_base_name + '.wav'
    sample_rate, samples = wavfile.read(audio_path)
    
    # Load the annotation file (typically .txt or .csv)
    annot_file_path = file_base_name + '.Table.1.selections.txt'
    df = pd.read_csv(annot_file_path, sep='\t')
    
    return sample_rate, samples, df

def extract_audio_segments(samples, sample_rate, annotations):
    segments = []
    for _, row in annotations.iterrows():
        start_time, end_time = row['Begin Time (s)'], row['End Time (s)']
        start_sample, end_sample = int(start_time * sample_rate), int(end_time * sample_rate)
        
        # Check if start and end indices are valid
        if start_sample >= end_sample or start_sample < 0 or end_sample > len(samples):
            print(f"Invalid segment: {start_time}-{end_time}s")
            continue
        
        segment = samples[start_sample:end_sample]
        segments.append({
            'Selection': row['Selection'],
            'Annotation': row['Annotation'],
            'Begin Time (s)': start_time,
            'End Time (s)': end_time,
            'audio_segment': segment,
            'Low Freq (Hz)': row['Low Freq (Hz)'],
            'High Freq (Hz)': row['High Freq (Hz)']
        })
    return segments

def generate_spectrogram(audio_segment, sample_rate, fmin=20, fmax=1000, nperseg=1024, nfft=2048, noverlap=512):  
    frequencies, times, spectrogram_data = signal.spectrogram(
        audio_segment, sample_rate, nperseg=nperseg, nfft=nfft, noverlap=noverlap, window='hann'
    )
    spectrogram_data = np.maximum(spectrogram_data, 1e-10)
    freq_slice = np.where((frequencies >= fmin) & (frequencies <= fmax))
    return frequencies[freq_slice], times, spectrogram_data[freq_slice]

def pad_or_crop_spectrogram(spectrogram_data, target_shape):
    # Ensure spectrogram data is a numpy array
    spectrogram_data = np.array(spectrogram_data)

    # Check if the spectrogram is 2D (frequency x time)
    if spectrogram_data.ndim == 2:
        current_shape = spectrogram_data.shape
        padded_spectrogram = np.zeros(target_shape)

        # Check if the current spectrogram size is within bounds
        for i in range(min(current_shape[0], target_shape[0])):  # Frequencies
            for j in range(min(current_shape[1], target_shape[1])):  # Time steps
                padded_spectrogram[i, j] = float(spectrogram_data[i, j])
        return padded_spectrogram
    else:
        print(f"Error: Spectrogram data has invalid dimensions {spectrogram_data.shape}. Expected 2D.")
        raise ValueError(f"Spectrogram data must be 2D with dimensions (frequency, time), but got shape {spectrogram_data.shape}.")

def create_no_call_segments(samples, sample_rate, call_annotations, silence_duration_threshold=0.5):
    # Get the end times of existing calls
    call_end_times = call_annotations['End Time (s)'].values
    call_start_times = call_annotations['Begin Time (s)'].values
    
    # Sort calls by start times (just in case)
    sorted_start_times = np.sort(call_start_times)
    sorted_end_times = call_end_times[np.argsort(call_start_times)]
    
    # Generate the 'no-call' periods by looking at gaps between successive call end times and next start times
    no_call_segments = []
    for i in range(len(sorted_start_times) - 1):
        end_of_previous_call = sorted_end_times[i]
        start_of_next_call = sorted_start_times[i + 1]
        
        # Check for gaps larger than the silence threshold
        if start_of_next_call - end_of_previous_call > silence_duration_threshold:
            # Append the no-call segment between two calls
            start_sample = int(end_of_previous_call * sample_rate)
            end_sample = int(start_of_next_call * sample_rate)
            no_call_segment = samples[start_sample:end_sample]
            no_call_segments.append(no_call_segment)
    
    # Optionally, add the silent period after the last call
    last_call_end_time = sorted_end_times[-1]
    audio_duration = len(samples) / sample_rate
    if audio_duration - last_call_end_time > silence_duration_threshold:
        start_sample = int(last_call_end_time * sample_rate)
        no_call_segment = samples[start_sample:]
        no_call_segments.append(no_call_segment)

    return no_call_segments

def get_category_name(annotation):
    category_mapping = {
        'Rupe A': 'rupe_A',
        'Rupe B': 'rupe_B',
        'Rupe C': 'rupe_C',
        'Growl B': 'growl_B',
        'Moan': 'moan',
        'G rupe': 'g_rupe',
        'Guttural rupe': 'g_rupe',
        'no_call': 'no_call',  
        'Type 4 A': 'type_4_A'
    }
    # Logging the unknown category
    if annotation not in category_mapping:
        print(f"Unknown annotation found: {annotation}")
    return category_mapping.get(annotation, 'unknown')

def calculate_baselines(folder_paths):
    """
    Calculate the baseline segment duration and frequency range across all files in the folder paths.
    """
    global max_time, max_frequency
    
    max_time, max_frequency = 0, 0
    
    # Iterate through each folder in the folder_paths list
    for folder_path in folder_paths:
        # Loop through each file in the current folder
        for file in os.listdir(folder_path):
            if file.endswith('.wav'):
                file_base_name = os.path.splitext(file)[0]
                file_base_path = os.path.join(folder_path, file_base_name)
                
                # Load the audio and annotation
                sample_rate, samples, df = load_audio_and_annotation(file_base_path)
                
                # Extract segments for the current file
                segments = extract_audio_segments(samples, sample_rate, df)
                
                # Find the longest duration in terms of time
                for segment in segments:
                    max_time = max(max_time, segment['End Time (s)'] - segment['Begin Time (s)'])
                    max_frequency = max(max_frequency, segment['High Freq (Hz)'] - segment['Low Freq (Hz)'])

# Folder paths for the WAV and annotation files
folder_paths = [
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Guttural rupe',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Rupes A and B',
    r'C:\Users\Admin\Downloads\Samples Grey Seal\Moan'
]

dataset = []

# Calculate the baseline values for time and frequency
calculate_baselines(folder_paths)

# Now, max_time and max_frequency contain the baseline values
target_shape = (int(max_frequency), int(max_time))

# Process audio files and annotations
for folder_path in folder_paths:
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_base_name = os.path.splitext(file)[0]
            file_base_path = os.path.join(folder_path, file_base_name)
            sample_rate, samples, df = load_audio_and_annotation(file_base_path)
            segments = extract_audio_segments(samples, sample_rate, df)

            # Generate no_call segments from silent gaps
            no_call_segments = create_no_call_segments(samples, sample_rate, df)
            
            # Combine call and no-call segments
            for audio_segment in no_call_segments:
                segments.append({
                    'Selection': 'no_call',  # Use 'no_call' as the selection for silent segments
                    'Annotation': 'no_call',
                    'Begin Time (s)': 0,
                    'End Time (s)': len(audio_segment) / sample_rate,
                    'audio_segment': audio_segment,
                    'Low Freq (Hz)': 0,
                    'High Freq (Hz)': sample_rate // 2
                })
            
            # Categorize segments and add to the dataset
            for segment in segments:
                category = get_category_name(segment['Annotation'])
                segment['Category'] = category
                dataset.append(pd.DataFrame([segment]))

# Concatenate all segment data into a single DataFrame
final_df = pd.concat(dataset, ignore_index=True)

# Save spectrograms for all segments
for _, segment_info in final_df.iterrows():
    audio_segment = segment_info['audio_segment']
    frequencies, times, spectrogram_data = generate_spectrogram(
        audio_segment, sample_rate, fmin=segment_info['Low Freq (Hz)'], fmax=segment_info['High Freq (Hz)']
    )
    
    # Pad or crop the spectrogram to match the maximum size
    try:
        padded_spectrogram = pad_or_crop_spectrogram(spectrogram_data, target_shape)
    except ValueError as e:
        print(f"Error while processing {segment_info['Selection']} ({segment_info['Category']}): {e}")
        continue

    # Save spectrogram to an NPZ file
    npz_filename = f"{segment_info['Category']}_{segment_info['Begin Time (s)']}_{segment_info['End Time (s)']}.npz"
    npz_file_path = os.path.join(output_dir, npz_filename)
    np.savez(npz_file_path, spectrogram_data=padded_spectrogram, frequencies=frequencies, times=times)


Unknown annotation found: nan
Unknown annotation found: nan
Unknown annotation found: Type 4 B
Unknown annotation found: unidentified
Unknown annotation found: Trrot
Unknown annotation found: Unidentified
Unknown annotation found: Trrot
Unknown annotation found: Trrot
Unknown annotation found: Trrot
Unknown annotation found: ??
Unknown annotation found: ?
Unknown annotation found: ?
Unknown annotation found: ??
Unknown annotation found: HS Groan
Unknown annotation found: HS Groan


In [2]:
def load_dataset(output_dir, final_df):
    """
    Loads spectrogram data and their corresponding labels from a directory containing .npz files,
    linking them to the information in final_df created in code 1.

    Parameters:
        - output_dir (str): Directory containing the .npz files.
        - final_df (pd.DataFrame): The DataFrame with category and annotation info.

    Returns:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    spectrograms = []
    labels = []

    # List unique categories present in final_df for reference
    available_categories = final_df["Category"].unique()

    # Loop through each .npz file in the folder
    for file in os.listdir(output_dir):
        if file.endswith(".npz"):
            try:
                # Load the .npz file
                data = np.load(os.path.join(output_dir, file))
                spectrogram_data = data["spectrogram_data"]
                
                # Extract the annotation (full category) from the filename
                file_name_parts = file.split('_')  # Split by underscore
                
                # Identify the category using parts of the filename (e.g., g_rupe or rupe_A)
                category_label = '_'.join(file_name_parts[:2])  # Grab the first two parts (e.g., g_rupe or rupe_A)

                # Find the corresponding row in final_df using the parsed label
                segment_row = final_df[final_df['Category'] == category_label]
                
                if not segment_row.empty:
                    category = segment_row.iloc[0]['Category']
                else:
                    category = 'unknown'
                
                # Skip files with 'unknown' categories
                if category != 'unknown':
                    spectrograms.append(spectrogram_data)
                    labels.append(category)

            except (KeyError, Exception):
                continue

    spectrograms = np.array(spectrograms, dtype=object)  # Use dtype=object for variable-length data
    labels = np.array(labels)

    return spectrograms, labels


def summarize_data(spectrograms, labels):
    """
    Summarizes the loaded spectrograms and their corresponding labels.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.
    """
    labels_df = pd.DataFrame(labels, columns=["Category"])

    num_samples = len(spectrograms)
    unique_labels = labels_df["Category"].nunique()
    category_counts = labels_df["Category"].value_counts()

    print(f"Total number of samples: {num_samples}")
    print(f"Unique categories: {unique_labels}")
    print("\nCategory distribution:")
    print(category_counts)

output_dir = "C:/Users/Admin/Downloads/Machine-Learning/Project_part2/spectrogram_output"  

# Example usage
spectrograms, labels = load_dataset(output_dir, final_df)
summarize_data(spectrograms, labels)


NameError: name 'final_df' is not defined

In [1]:
def preprocess_data(spectrograms, labels):
    """
    Preprocess the spectrogram data by normalizing and splitting it into training and testing sets.

    Parameters:
        - spectrograms (np.ndarray): Array of spectrogram data.
        - labels (np.ndarray): Array of corresponding labels.

    Returns:
        - X_train, X_test (np.ndarray): Normalized training and testing spectrogram data.
        - y_train, y_test (np.ndarray): Corresponding labels for training and testing data.
    """
    # Flatten spectrogram data to 2D array
    X = np.array([spec.flatten() for spec in spectrograms], dtype=np.float32)
    y = np.array(labels, dtype=np.int32)

    # Normalize the data
    scaler = MinMaxScaler(feature_range=(0, 1))  # Normalize between 0 and 1
    X_normalized = scaler.fit_transform(X)

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test

# Step 1: Load the dataset
spectrograms, labels = load_dataset(output_dir, final_df)

# Step 2: Summarize the loaded data
summarize_data(spectrograms, labels)

# Step 3: Preprocess the data
X_train, X_test, y_train, y_test = preprocess_data(spectrograms, labels)

# Output the shapes of the data after preprocessing
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

NameError: name 'load_dataset' is not defined